In [14]:
import os
import requests
import zipfile
import io
import glob
import subprocess
import time
import sys

In [2]:
def download_nist_test(test_num, dataset_id, custom_folder):
    """
    Downloads and extracts NIST wind data directly from Amazon S3.
    """
    # Construct the Direct URL using the pattern we found
    url = f"https://s3.amazonaws.com/nist-el/resilience_multihazards/winddata/{test_num}/{dataset_id}/{dataset_id}.zip"
    
    # Define the download path
    if not os.path.exists(custom_folder):
        os.makedirs(custom_folder)
    
    print(f"\nTargeting: {url}")
    print(f"Destination: {os.path.abspath(custom_folder)}")

    try:
        # 1. Start the download stream
        response = requests.get(url, stream=True)
        response.raise_for_status() # This catches 404s if the ID is wrong

        # 2. Extract in-memory (no need to save the .zip file to disk)
        print("Downloading and Unzipping...")
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            z.extractall(custom_folder)
            
        print(f"Success! {len(os.listdir(custom_folder))} files extracted.")
        return True

    except requests.exceptions.HTTPError:
        print(f"Error: Could not find '{dataset_id}' in '{test_num}'. Check your spelling!")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    return False

# --- User Input ---
print("--- NIST Wind Data Automation Tool ---")
test = input("Enter Test Number (e.g., ss20-test1): ").strip()
ds_id = input("Enter Data Set ID (e.g., he1): ").strip()
folder = input("Enter folder name for these files: ").strip()

if download_nist_test(test, ds_id, folder):
    print("\nNext step: Run the NIST Decoder.")

--- NIST Wind Data Automation Tool ---

Targeting: https://s3.amazonaws.com/nist-el/resilience_multihazards/winddata/ss20-test1/he1/he1.zip
Destination: c:\WINDLAB_SUMMER\TestDir
Success! 37 files extracted.

Next step: Run the NIST Decoder.


In [4]:
def extract_angle_data(directory, target_angle):
    """
    Finds and extracts the HDF from a specific angle ZIP file.
    Example: target_angle = 195.0
    """
    # 1. Format the angle to match the filename (195.0 -> '1950')
    angle_str = str(int(target_angle * 10)).zfill(4)
    search_pattern = f"*a{angle_str}.zip"
    
    # 2. Search for the file in the directory
    matching_files = glob.glob(os.path.join(directory, search_pattern))
    
    if not matching_files:
        print(f"Error: No ZIP file found for angle {target_angle}")
        return None

    zip_path = matching_files[0]
    print(f"Found: {os.path.basename(zip_path)}")

    # 3. Extract the HDF
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Get the name of the HDF file inside the zip
        hdf_filename = [f for f in z.namelist() if f.endswith('.HDF')][0]
        z.extract(hdf_filename, directory)
        print(f"Extracted: {hdf_filename}")
        return os.path.join(directory, hdf_filename)

# --- Execution ---
user_angle = float(input("Enter desired wind angle (e.g., 195.0): "))
extracted_hdf = extract_angle_data("TestDir", user_angle)

Found: ADW100o100L016a1800.zip
Extracted: ADW100o100L016a1800.HDF


In [23]:
def run_nist_decoder_clean(hdf_path, exe_path):
    # 1. Absolute path, strict backslashes, NO trailing spaces
    hdf_path = os.path.abspath(hdf_path).replace("/", "\\").strip()
    
    process = subprocess.Popen(
        [exe_path],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=0 # Unbuffered to prevent character lag
    )

    try:
        # 2. Send filename. 
        # Using write() without \n first, then flushing, then sending the \n
        # sometimes helps legacy C++ tools distinguish the two.
        process.stdin.write(hdf_path)
        process.stdin.write("\n")
        process.stdin.flush()
        
        # 3. Wait and monitor
        while True:
            line = process.stdout.readline()
            if not line: break
            print(f"DECODER: {line.strip()}")
            
            if "Enter a or n or s" in line:
                time.sleep(1) # Longer pause to let the buffer clear
                process.stdin.write("a\n")
                process.stdin.flush()
                break
        
        process.communicate(timeout=10)
        return True
    except Exception as e:
        print(f"Failed: {e}")
        return False

# Execution
exe_path = r"c:\WINDLAB\NIST_Download\bin\win32\hdf_read_demo_win.exe"
hdf_file = r"C:\WINDLAB_SUMMER\TestDir\ADW100o100L016a1800.HDF"

run_nist_decoder_clean(hdf_file, exe_path)

DECODER: 
DECODER: infile =
DECODER: 
DECODER: C:\WINDLAB_SUMMER\TestDir\ADW100o100L016a1800.HDF
DECODER: 
DECODER: 
DECODER: Enter <a> to load time histories from ALL taps
DECODER: Enter <n> to load NO time history data (supporting information only)
DECODER: Enter <s> to load time hitories from a SUBSET of taps and points
DECODER: Enter a or n or s :ERROR: Enter s or a or n
Failed: [Errno 22] Invalid argument


False